In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


# Bronze to Silver Transform
## Purpose
Reads from bronze_orders (raw Delta table) and writes a clean, standardised
silver_orders table. This notebook is the single source of truth for data cleaning.
## Rules applied
- Drop rows where Order_ID or Sales is null
- Standardise column names to snake_case
- Cast Sales and Profit to DoubleType
- Parse Order_Date and Ship_Date as DateType
- Trim whitespace from all string columns
- Add a processed_timestamp column

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType
df_bronze = spark.table('bronze_orders')
print(f'Bronze row count : {df_bronze.count()}')
print(f'Bronze columns : {len(df_bronze.columns)}')
df_bronze.printSchema()

StatementMeta(, dd6865f8-0600-40d0-8083-cea2ed155a5f, 11, Finished, Available, Finished, False)

Bronze row count : 10194
Bronze columns : 21
root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State/Province: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [9]:

from pyspark.sql.functions import col, count, when, isnan
null_counts = df_bronze.select([
count(when(col(c).isNull() | (col(c) == ''), c)).alias(c)
for c in df_bronze.columns
])
null_counts.show(vertical=True)

StatementMeta(, dd6865f8-0600-40d0-8083-cea2ed155a5f, 12, Finished, Available, Finished, False)

-RECORD 0-------------
 Row_ID         | 0   
 Order_ID       | 0   
 Order_Date     | 0   
 Ship_Date      | 0   
 Ship_Mode      | 0   
 Customer_ID    | 0   
 Customer_Name  | 0   
 Segment        | 0   
 Country/Region | 0   
 City           | 0   
 State/Province | 0   
 Postal_Code    | 0   
 Region         | 0   
 Product_ID     | 0   
 Category       | 0   
 Sub-Category   | 0   
 Product_Name   | 0   
 Sales          | 0   
 Quantity       | 0   
 Discount       | 0   
 Profit         | 0   



In [10]:
from pyspark.sql.functions import trim, to_date, current_timestamp
# Step A: rename columns to snake_case (adjust to match YOUR column names)
rename_map = {
'Order ID' : 'order_id',
'Order Date' : 'order_date',
'Ship Date' : 'ship_date',
'Ship Mode' : 'ship_mode',
'Customer ID': 'customer_id',
'Customer Name': 'customer_name',
'Segment' : 'segment',
'Country' : 'country',
'City' : 'city',
'State' : 'state',
'Postal Code': 'postal_code',
'Region' : 'region',
'Product ID' : 'product_id',
'Category' : 'category',
'Sub-Category': 'sub_category',
'Product Name': 'product_name',
'Sales' : 'sales',
'Quantity' : 'quantity',
'Discount' : 'discount',
'Profit' : 'profit',
'Order_ID' : 'order_id', # handle underscore variants
}
df = df_bronze
for old, new in rename_map.items():
   if old in df.columns:
       df = df.withColumnRenamed(old, new)

# Step B: trim whitespace on all string columns
string_cols = [f.name for f in df.schema.fields if str(f.dataType) ==
'StringType()']
for c in string_cols:
    df = df.withColumn(c, trim(col(c)))
# Step C: drop rows where order_id or sales is null/blank
df = df.filter(
col('order_id').isNotNull() & (col('order_id') != '') &
col('sales').isNotNull()
)
# Step D: cast numeric columns
df = df.withColumn('sales', col('sales').cast(DoubleType()))
df = df.withColumn('profit', col('profit').cast(DoubleType()))
df = df.withColumn('discount', col('discount').cast(DoubleType()))
df = df.withColumn('quantity', col('quantity').cast('integer'))
# Step E: parse dates
df = df.withColumn('order_date', to_date(col('order_date'), 'M/d/yyyy'))
df = df.withColumn('ship_date', to_date(col('ship_date'), 'M/d/yyyy'))
# Step F: add audit column
df = df.withColumn('processed_timestamp', current_timestamp())
print(f'Silver row count : {df.count()}')
df.printSchema()
df.show(5, truncate=False)

StatementMeta(, dd6865f8-0600-40d0-8083-cea2ed155a5f, 13, Finished, Available, Finished, False)

Silver row count : 10194
root
 |-- Row_ID: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- State/Province: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- processed_timestamp: timestamp (nullable = false)

+------+--------------+----------+----------+--------

In [11]:
df.write.mode('overwrite') \
.format('delta') \
.saveAsTable('silver_orders')
print('silver_orders written successfully')

StatementMeta(, dd6865f8-0600-40d0-8083-cea2ed155a5f, 14, Finished, Available, Finished, False)

silver_orders written successfully


## Section 2 - Bronze Targets to Silver Targets
Reads bronze_targets (loaded by the Pipeline from sales_targets.xlsx)
and writes a clean silver_targets Delta table.


In [12]:
df_targets = spark.table('bronze_targets')
print(f'Targets row count: {df_targets.count()}')
df_targets.printSchema()
df_targets.show()

StatementMeta(, dd6865f8-0600-40d0-8083-cea2ed155a5f, 15, Finished, Available, Finished, False)

Targets row count: 11
root
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Sales_Target: string (nullable = true)

+-------+---------------+-------+------------+
| Region|       Category|  Month|Sales_Target|
+-------+---------------+-------+------------+
|   West|     Technology|2024-01|       45000|
|   West|      Furniture|2024-01|       28000|
|   West|Office Supplies|2024-01|       19000|
|   East|     Technology|2024-01|       52000|
|   East|      Furniture|2024-01|       31000|
|   East|Office Supplies|2024-01|       22000|
|Central|     Technology|2024-01|       38000|
|Central|      Furniture|2024-01|       24000|
|Central|Office Supplies|2024-01|       17000|
|  South|     Technology|2024-01|       29000|
|  South|      Furniture|2024-01|       18000|
+-------+---------------+-------+------------+



In [13]:
from pyspark.sql.functions import trim, col, current_timestamp
from pyspark.sql.types import DoubleType

df_t = df_targets \
.withColumnRenamed('Sales_Target', 'sales_target') \
.withColumn('region', trim(col('Region'))) \
.withColumn('category', trim(col('Category'))) \
.withColumn('month', trim(col('Month'))) \
.withColumn('sales_target', col('Sales_Target').cast(DoubleType())) \
.withColumn('processed_timestamp', current_timestamp()) \
.select('region', 'category', 'month', 'sales_target', 'processed_timestamp')
df_t.write.mode('overwrite').format('delta').saveAsTable('silver_targets')
print(f'silver_targets written: {df_t.count()} rows')

StatementMeta(, dd6865f8-0600-40d0-8083-cea2ed155a5f, 17, Finished, Available, Finished, False)

silver_targets written: 11 rows


In [14]:
# Verify processed_timestamp data type
spark.table("silver_orders") \
    .select("processed_timestamp") \
    .printSchema()

spark.table("silver_orders") \
    .select("processed_timestamp") \
    .show(3, truncate=False)

StatementMeta(, dd6865f8-0600-40d0-8083-cea2ed155a5f, 18, Finished, Available, Finished, False)

root
 |-- processed_timestamp: timestamp (nullable = true)

+--------------------------+
|processed_timestamp       |
+--------------------------+
|2026-07-08 11:02:22.037027|
|2026-07-08 11:02:22.037027|
|2026-07-08 11:02:22.037027|
+--------------------------+
only showing top 3 rows

